> **Nota:** Este notebook documenta el análisis de viabilidad inicial de fuentes de datos. 
La arquitectura final ha evolucionado para incluir enriquecimiento con IA local (Ollama). 
Los problemas de moneda detectados posteriormente (Polonia usa PLN, no EUR) tampoco están reflejados aquí. 
Ver [README](../README.md) para el estado actual del proyecto.

# TechRadar — Viabilidad de Fuentes de Datos
## Notebook 00: EDA y decisiones por fuente

**Objetivo:** Dashboard del mercado tech en Europa.  
**KPIs:** Top skills, Salarios, Distribución por país, Remote %, Tendencias

---

### Fuentes seleccionadas

| Fuente | Decisión | Motivo |
|--------|----------|--------|
| **Adzuna** | ✅ MANTENER | Salario numérico, 8 países EU, redirect_url con texto completo |
| **Eurostat** | ✅ MANTENER (contexto) | Datos oficiales UE, 0 nulos, CC BY 4.0 |

### Fuentes descartadas

| Fuente | Motivo de descarte |
|--------|-------------------|
| Remotive | Ubicaciones no parseables, categorías rotas, salario inutilizable, 100% remote |
| The Muse | Cero ofertas europeas, sin salario, categorías incorrectos |
| InfoJobs | Bloqueado Incapsula/Imperva HTTP 405 |
| LinkedIn | ToS prohíbe scraping, riesgo legal |
| Indeed ES | Cloudflare 403 total |
| Tecnoempleo | Cloudflare 403 |
| Jooble | Cloudflare 403 |
| Arbeitnow | 90% ofertas de Alemania — sesgo geográfico |
| Jobicy | 70% ofertas de USA — sesgo geográfico |
| Reed.co.uk | Solo UK (no UE), GBP ≠ EUR |
| UK (Adzuna/gb) | UK no pertenece a la UE, GBP ≠ EUR, sin datos Eurostat post-Brexit |

---

### Solución al truncado de Adzuna

La `redirect_url` apunta siempre a `www.adzuna.es` (dominio propio).  
Crawling verificado: 4/5 éxito, 5x–16x más texto. Sin anti-bot, sin riesgo legal.

## Imports y configuración

In [1]:
import json
import re
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

RAW_DIR = Path("../data/raw_samples")

print("Muestras disponibles:")
for f in sorted(RAW_DIR.glob("*.json")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:40s}  {size_kb:.1f} KB")

Muestras disponibles:
  adzuna_sample.json                        8.2 KB
  eurostat_sample.json                      1.1 KB


---
## 1. Adzuna API — MANTENER (fuente principal)

**Auth:** App ID + App Key (gratuito en https://developer.adzuna.com/)  
**Endpoint:** `GET /v1/api/jobs/{country}/search/{page}`  
**Cobertura:** DE, FR, ES, NL, PL, IT, AT, BE (8 países EU — PL usa PLN, los 7 restantes son eurozona)

In [2]:
adzuna_path = RAW_DIR / "adzuna_sample.json"
adzuna_raw = json.loads(adzuna_path.read_text(encoding="utf-8"))

if "error" in adzuna_raw:
    print(f"Estado: {adzuna_raw['error']}")
    print("Configura .env con ADZUNA_APP_ID y ADZUNA_APP_KEY")
    adzuna_available = False
else:
    adzuna_records = adzuna_raw.get("results", [])
    adzuna_df = pd.DataFrame(adzuna_records)
    adzuna_available = True
    print(f"Registros: {len(adzuna_df)}")
    print(f"Columnas: {list(adzuna_df.columns)}")

Registros: 5
Columnas: ['longitude', 'adref', 'company', 'created', '__CLASS__', 'salary_is_predicted', 'location', 'description', 'category', 'latitude', 'redirect_url', 'id', 'title']


In [3]:
if adzuna_available:
    print("=== TIPOS ===")
    print(adzuna_df.dtypes)
    print()

    print("=== NULOS (%) ===")
    null_pct = (adzuna_df.isnull().sum() / len(adzuna_df) * 100).round(1)
    has_nulls = null_pct[null_pct > 0]
    print(has_nulls.to_string() if not has_nulls.empty else "Sin nulos")
    print()

    key_cols = ["title", "company", "location", "salary_min",
                "salary_max", "created", "redirect_url"]
    available_cols = [c for c in key_cols if c in adzuna_df.columns]
    display(adzuna_df[available_cols].head(3))
    print()

    if "salary_min" in adzuna_df.columns:
        print("=== SALARIO ===")
        print(f"salary_min tipo: {adzuna_df['salary_min'].dtype}, "
              f"nulos: {adzuna_df['salary_min'].isnull().sum()}")
        if adzuna_df['salary_min'].notna().any():
            print(f"Rango: {adzuna_df['salary_min'].min():.0f}"
                  f" - {adzuna_df['salary_max'].max():.0f}")
        if "salary_is_predicted" in adzuna_df.columns:
            pred = (adzuna_df['salary_is_predicted'] == '1').sum()
            print(f"salary_is_predicted=1: {pred}/{len(adzuna_df)}")
    else:
        print("=== SALARIO ===")
        print("salary_min / salary_max no presentes en esta muestra.")
        print("Cobertura salarial baja en Alemania (~30%). Mayor en ES, FR, NL.")
        print("Ver sección 1c para el análisis completo de salarios.")
    print()

    if "description" in adzuna_df.columns:
        desc_len = adzuna_df['description'].dropna().str.len()
        print("=== DESCRIPCION ===")
        print(f"Media: {desc_len.mean():.0f} | Min: {desc_len.min()} | Max: {desc_len.max()}")
        if desc_len.nunique() == 1:
            print(f"ALERTA: truncado a exactamente {desc_len.iloc[0]} chars.")
            print("Límite del plan gratuito de la API.")

=== TIPOS ===
longitude              float64
adref                   object
company                 object
created                 object
__CLASS__               object
salary_is_predicted     object
location                object
description             object
category                object
latitude               float64
redirect_url            object
id                      object
title                   object
dtype: object

=== NULOS (%) ===
Sin nulos



,title,company,location,created,redirect_url
0,Data Engineer - Platform Engineering,"{'display_name': 'N26 GmbH', '__CLASS__': 'Adzuna::API::Response::Company'}","{'__CLASS__': 'Adzuna::API::Response::Location', 'display_name': 'Berlin, De...",2026-05-05T16:42:32Z,https://www.adzuna.de/land/ad/5719528628?se=zrAILr5I8RGsLNZFFTop6g&utm_mediu...
1,Customer Data Engineer (w/m/d),"{'display_name': 'MAINGAU Energie GmbH', '__CLASS__': 'Adzuna::API::Response...","{'display_name': 'Obertshausen, Offenbach (Kreis)', '__CLASS__': 'Adzuna::AP...",2026-05-02T16:51:20Z,https://www.adzuna.de/land/ad/5717126194?se=zrAILr5I8RGsLNZFFTop6g&utm_mediu...
2,Senior Data Engineer (w/m/d),"{'__CLASS__': 'Adzuna::API::Response::Company', 'display_name': 'WUND HOLDIN...","{'area': ['Deutschland', 'Baden-Württemberg', 'Rhein-Neckar-Kreis', 'Sinshei...",2026-05-03T16:41:30Z,https://www.adzuna.de/land/ad/5717900912?se=zrAILr5I8RGsLNZFFTop6g&utm_mediu...



=== SALARIO ===
salary_min / salary_max no presentes en esta muestra.
Cobertura salarial baja en Alemania (~30%). Mayor en ES, FR, NL.
Ver sección 1c para el análisis completo de salarios.

=== DESCRIPCION ===
Media: 500 | Min: 500 | Max: 500
ALERTA: truncado a exactamente 500 chars.
Límite del plan gratuito de la API.


### 1a. Solución al truncado — Crawling de redirect_url ✅ VERIFICADO

El campo `redirect_url` apunta siempre a `www.adzuna.es`. No hay anti-bot.

| Oferta | Chars API | Chars crawling | Multiplicador |
|--------|-----------|----------------|--------------|
| Senior Data Engineer | 500 | 2.367 | 5x |
| Freelance Data Scraping | 500 | 3.984 | 8x |
| Data Engineer Datavant | 500 | 8.384 | 16x |
| Data Science Engineer | 500 | 4.405 | 9x |

**Skills extraídas del texto completo (ejemplo):**  
Python, SQL, ETL/ELT, Airflow, Databricks, RAG, embeddings,  
vector databases, Azure, Snowflake, Oracle, Kafka, Kubernetes, LLM.

Con los 500 chars de la API no habría podido extraerse ninguna de esas tecnologías.

### 1b. Volumen y límites del plan gratuito

| País | IT jobs disponibles | Extraíbles (free) |
|------|---------------------|-------------------|
| Alemania | 27.967 | 5.000 |
| Francia | 34.438 | 5.000 |
| Polonia | 43.441 | 5.000 |
| Italia | 9.168 | 5.000 |
| Países Bajos | 5.319 | 5.000 |
| España | 2.764 | 2.764 |
| Bélgica | 1.333 | 1.333 |
| Austria | 1.179 | 1.179 |
| **TOTAL** | **125.609** | **30.276** |

**Límites ToS oficiales:** 25/min, 250/día, 1.000/semana, **2.500/mes**

- Carga inicial: ~606 llamadas (30.276 / 50 por página)
- Actualizaciones semanales con `max_days_old=7`: ~60-80 llamadas
- Crawling de `redirect_url` **no cuenta** contra la cuota de la API
- Uso personal/portfolio permitido según ToS

### 1c. Análisis de salarios — outliers y decisión sobre `salary_mid`

**Problema:** Adzuna devuelve `salary_min` y `salary_max` por separado.  
Para el dashboard necesitamos un único valor de referencia por oferta.

**Decisión:** `salary_mid = (salary_min + salary_max) / 2`  
El punto medio del rango es el estimador individual más neutro. Se calcula en `transform.py` y se persiste en la tabla `jobs`.

**¿Media o mediana para los agregados del dashboard?**  
Los salarios tienen distribución asimétrica positiva (pocos roles muy bien pagados tiran la media hacia arriba). Además, en datos de job boards existen outliers artificiales habituales:
- Salarios mensuales introducidos donde debería ir el anual (€2.500 en vez de €30.000)
- Salarios anuales en el campo equivocado (€120.000 como salary_min cuando es el máximo real)
- Predicciones algorítmicas de Adzuna (`salary_is_predicted = True`) con errores sistemáticos

**Conclusión: se usa la mediana** en todas las vistas de agregación del dashboard.  
La mediana es resistente a outliers. Es también el estándar de Eurostat, Glassdoor y LinkedIn para reportar salarios de mercado.

**Cobertura salarial por país (estimada):**
- Alemania (~30%): las empresas alemanas tienden a no publicar salario
- España, Francia, Países Bajos (~50-70%): mayor transparencia salarial
- El análisis de outliers se ejecuta sobre el dataset completo en `transform.py`

In [4]:
# Análisis de cobertura y outliers salariales
# Este código se ejecuta sobre la muestra disponible y documenta
# el enfoque que se aplicará al dataset completo (~30.000 ofertas).

if adzuna_available and "salary_min" in adzuna_df.columns:
    salary_df = adzuna_df[["title", "salary_min", "salary_max",
                            "salary_is_predicted"]].copy()
    salary_df["salary_min"] = pd.to_numeric(salary_df["salary_min"], errors="coerce")
    salary_df["salary_max"] = pd.to_numeric(salary_df["salary_max"], errors="coerce")

    con_salario = salary_df["salary_min"].notna()
    cobertura = con_salario.sum() / len(salary_df) * 100
    print(f"Cobertura salarial: {con_salario.sum()}/{len(salary_df)} ({cobertura:.0f}%)")
    print()

    salary_df["salary_mid"] = (
        (salary_df["salary_min"] + salary_df["salary_max"]) / 2
    ).round(0).astype("Int64")

    s = salary_df.loc[con_salario, "salary_mid"]
    print("=== DISTRIBUCIÓN salary_mid ===")
    print(f"  Media:   {s.mean():>10,.0f} EUR")
    print(f"  Mediana: {s.median():>10,.0f} EUR  ← métrica de referencia")
    print(f"  P10:     {s.quantile(0.10):>10,.0f} EUR")
    print(f"  P90:     {s.quantile(0.90):>10,.0f} EUR")
    print(f"  Min:     {s.min():>10,.0f} EUR")
    print(f"  Max:     {s.max():>10,.0f} EUR")
    print()

    # Umbrales de outlier para ofertas EU en EUR (anuales)
    OUTLIER_MIN = 12_000   # < 12k EUR/año → probable salario mensual mal introducido
    OUTLIER_MAX = 300_000  # > 300k EUR/año → probable error de entrada

    outliers_bajos = (s < OUTLIER_MIN).sum()
    outliers_altos = (s > OUTLIER_MAX).sum()
    print(f"Outliers detectados (< {OUTLIER_MIN:,} EUR): {outliers_bajos}")
    print(f"Outliers detectados (> {OUTLIER_MAX:,} EUR): {outliers_altos}")
    print()
    print("Estos registros se marcarán pero NO se eliminarán en transform.py.")
    print("Las vistas del dashboard filtran salary_is_predicted=FALSE")
    print("y usan mediana, lo que absorbe los outliers restantes.")

else:
    print("Muestra actual sin datos salariales (endpoint /de, cobertura ~30%).")
    print()
    print("El análisis completo se ejecutará sobre ~30.000 ofertas reales.")
    print("El código de arriba es el que se usará en transform.py.")
    print()
    print("Umbrales de outlier definidos para EUR anuales:")
    print("  < 12.000 EUR/año  → probable salario mensual mal introducido")
    print("  > 300.000 EUR/año → probable error de entrada")
    print()
    print("Decisión tomada: salary_mid = (salary_min + salary_max) / 2")
    print("Dashboard usa mediana de salary_mid, no media (robusta frente a outliers).")

Muestra actual sin datos salariales (endpoint /de, cobertura ~30%).

El análisis completo se ejecutará sobre ~30.000 ofertas reales.
El código de arriba es el que se usará en transform.py.

Umbrales de outlier definidos para EUR anuales:
  < 12.000 EUR/año  → probable salario mensual mal introducido
  > 300.000 EUR/año → probable error de entrada

Decisión tomada: salary_mid = (salary_min + salary_max) / 2
Dashboard usa mediana de salary_mid, no media (robusta frente a outliers).


**Resumen de decisiones sobre salarios:**

| Campo | Decisión | Motivo |
|-------|----------|--------|
| `salary_min` / `salary_max` | Conservar en BD | Datos originales sin alterar |
| `salary_mid` | Crear en `transform.py` | Valor único de referencia por oferta |
| Agregados dashboard | **Mediana** de `salary_mid` | Robusta frente a outliers |
| `salary_is_predicted` | Filtrar en vistas | Los predichos tienen más dispersión |
| Outliers extremos | Mantener, no borrar | La mediana los absorbe; el histórico es válido |

### Conclusiones Adzuna

| Aspecto | Evaluación |
|---------|------------|
| Salario | ✅ Numérico int64, salary_is_predicted disponible, salary_mid calculado en transform |
| Ubicación | ✅ Parseable, display_name + jerarquía geográfica |
| Fechas | ✅ ISO 8601 |
| Descripción truncada | ✅ Resuelto: crawling redirect_url → 5x-16x más texto |
| Cobertura | 8 países EU, 30.276 registros IT extraíbles |
| Rate limits | 2.500/mes: suficiente para carga + actualizaciones semanales |
| Riesgo | Muy bajo — API oficial, uso personal en ToS |

**Decisión: MANTENER**

---
## 2. Eurostat API — MANTENER (contexto macro)

**Auth:** Sin autenticación  
**Licencia:** CC BY 4.0  
**Dataset:** `lfsi_emp_a` — Tasas de empleo anuales 15-64 por país  
**Cobertura:** 8 países EU del proyecto (UK no está en Eurostat post-Brexit)

In [5]:
eurostat_path = RAW_DIR / "eurostat_sample.json"
eurostat_raw = json.loads(eurostat_path.read_text(encoding="utf-8"))

print(f"Dataset: {eurostat_raw['dataset']}")
print(f"Uso: {eurostat_raw['use_case']}")
print()
eurostat_df = pd.DataFrame(eurostat_raw["records"])
print(f"Registros: {len(eurostat_df)}")
display(eurostat_df)

Dataset: lfsi_emp_a
Uso: Datos contextuales del mercado laboral — NO ofertas de trabajo

Registros: 5


,country_code,year,indicator,value
0,Germany,2023,Employment rate 15-64 (% population),77.3
1,Spain,2023,Employment rate 15-64 (% population),65.3
2,France,2023,Employment rate 15-64 (% population),68.4
3,Netherlands,2023,Employment rate 15-64 (% population),82.4
4,Poland,2023,Employment rate 15-64 (% population),72.4


In [6]:
print("=== NULOS ===")
print(eurostat_df.isnull().sum().to_string())
print()

print("=== TASAS DE EMPLEO 2023 ===")
for _, row in eurostat_df.sort_values("value", ascending=False).iterrows():
    bar = chr(9608) * int(row["value"] / 5)
    print(f"  {row['country_code']:15s} {row['value']:.1f}%  {bar}")
print()

print("=== ESTADÍSTICAS ===")
print(eurostat_df["value"].describe().to_string())

=== NULOS ===
country_code    0
year            0
indicator       0
value           0

=== TASAS DE EMPLEO 2023 ===
  Netherlands     82.4%  ████████████████
  Germany         77.3%  ███████████████
  Poland          72.4%  ██████████████
  France          68.4%  █████████████
  Spain           65.3%  █████████████

=== ESTADÍSTICAS ===
count     5.000000
mean     73.160000
std       6.847116
min      65.300000
25%      68.400000
50%      72.400000
75%      77.300000
max      82.400000


### Conclusiones Eurostat

| Aspecto | Evaluación |
|---------|------------|
| Calidad | Muy alta — datos oficiales UE, 0 nulos |
| Uso en BD | Tabla `labor_market_context` separada |
| Licencia | CC BY 4.0 — sin restricciones |
| Riesgo | Ninguno |

**Decisión: MANTENER**

---
## 3. Arquitectura final del pipeline

```
EXTRACCIÓN (extract.py)
  Adzuna API → datos estructurados (it-jobs, 8 países EU, todos en EUR)
    + redirect_url → crawling www.adzuna.es → descripción completa para NLP
  Eurostat API → tasas de empleo por país/año

TRANSFORMACIÓN (transform.py)
  Normalización de campos
  Extracción de texto limpio del HTML crawleado
  salary_mid = (salary_min + salary_max) / 2   ← campo calculado
  NLP: skills de descripción completa
  NLP: role_category desde título
  Detección remote: palabras clave en título/descripción/ubicación

CARGA (load.py)
  PostgreSQL (Supabase) — UPSERT incremental por job.id
    jobs            (salarios en EUR, salary_mid incluido)
    skills          (catálogo normalizado)
    job_skills      (relación M:N, NLP)
    labor_market_context  (Eurostat)
  Tras cada carga: marcar is_active=FALSE en jobs con posted_at > 90 días
```

### Estrategia de carga incremental

La base de datos **acumula datos semana a semana**. Nunca se borran registros.

- **Carga inicial:** `max_days_old=30`, ~606 llamadas, ~30.000 ofertas
- **Cargas semanales:** `max_days_old=7`, ~60-80 llamadas, ~3.000 ofertas nuevas
- **UPSERT por `id`:** si la oferta ya existe, se actualiza `last_seen_at`; si es nueva, se inserta
- **`is_active`:** campo booleano que indica si la oferta está vigente
  - Se pone a `FALSE` automáticamente cuando `posted_at < NOW() - 90 días`
  - Las vistas del dashboard filtran por `is_active = TRUE` para el mercado actual
  - Las vistas de tendencias históricas no filtran por `is_active` (usan todo el histórico)

### Infraestructura de despliegue

| Componente | Servicio | Tier |
|------------|---------|------|
| Orquestación + cómputo | GitHub Actions (cron semanal, lunes 6AM UTC) | Gratuito |
| Base de datos | Supabase PostgreSQL | Gratuito 500MB |
| Almacenamiento raw | En memoria GitHub Actions | No persistente |

**Presupuesto mensual de API:** ~300-350 llamadas de 2.500 disponibles